# Nhận diện cấu trúc giải phẫu: chạy thí nghiệm trên Colab
**Chọn GPU trước:** Runtime → Change runtime type → GPU (T4 trở lên).
Chạy lần lượt từng cell. Nếu phiên bị ngắt: chạy lại Cell 1, rồi tiếp tục từ bước đang dở (dữ liệu đã xử lý và kết quả được lưu trên Drive).
Chép lại **nguyên văn** mọi lỗi/ output bất thường cho Claude. Không sửa số liệu bằng tay.

## Cell 1. Gắn Drive, đặt đường dẫn
Thư mục dự án trên Drive phải chứa `src/` (upload thư mục `nhandiencautrucgiaiphauvacongcu` lên `MyDrive/anatomy_project`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
P='/content/drive/MyDrive/anatomy_project'
os.environ.update(PROJECT_DIR=P, RESULTS_DIR=P+'/results', CACHE_DIR=P+'/cache', DATA_DIR='/content/data')
os.makedirs(P+'/cache', exist_ok=True); os.makedirs('/content/data', exist_ok=True)
%cd {P}/src

## Cell 2 (Bước 0). Cài thư viện + báo cáo môi trường
Colab đã có PyTorch+CUDA nên **không cài lại torch** và không dùng virtualenv (runtime Colab đã là môi trường cô lập). Nếu pip báo xung đột numpy: Runtime → Restart session rồi chạy lại từ Cell 1.

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!pip -q install monai nnunetv2 totalsegmentator segmentation-models-pytorch transformers simpleitk nibabel opencv-python-headless scikit-image scikit-learn pandas matplotlib onnx onnxruntime-gpu remotezip tabulate kaggle
!python env_report.py

## Cell 3 (Bước 1A). CT: AMOS, chỉ tập con 40 ca
Script in dung lượng cần tải, nguồn, giấy phép rồi **chờ bạn gõ `yes`**. Hãy kiểm tra link `amos22.zip` đúng trên trang chính thức AMOS (https://amos22.grand-challenge.org/) / Zenodo; nếu khác, thêm `--url <link>`.
Nếu không tải được, script báo lỗi: dừng lại và báo Claude (sẽ đề xuất BTCV hoặc tập khác).

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python data_ct.py --n 40

## Cell 4 (Bước 1B). CAMUS: tải thẳng vào Drive từ Colab
Nguồn chính thức: CREATIS (humanheart-project); đọc điều khoản sử dụng trên trang CAMUS. Server nén zip lúc tải nên không biết dung lượng trước và không resume được; `curl` sẽ tự thử lại cả file. Cell 4a thử tốc độ 30 giây (dữ liệu bỏ đi), **dán kết quả `speed` cho Claude** nếu thấy chậm (vài chục KB/s).

In [ ]:
!curl -sL --max-time 30 -o /dev/null -w "bytes=%{size_download} speed=%{speed_download}B/s\n" "https://humanheart-project.creatis.insa-lyon.fr/database/api/v1/collection/6373703d73e9f0047faa1bc8/download"

**Cell 4b**: tải vào `MyDrive/CAMUS_public.zip` (bỏ qua nếu file đã có). Có thể mất lâu; giữ tab mở.

In [ ]:
import os
Z='/content/drive/MyDrive/CAMUS_public.zip'
if os.path.exists(Z):
    print('Đã có', Z)
else:
    get_ipython().system(f'curl -L --retry 10 --retry-all-errors --retry-delay 5 --connect-timeout 30 -o {Z}.part "https://humanheart-project.creatis.insa-lyon.fr/database/api/v1/collection/6373703d73e9f0047faa1bc8/download" && mv {Z}.part {Z}')
print(os.path.exists(Z) and f'{os.path.getsize(Z)/2**20:.0f} MB')

**Cell 4c**: giải nén rồi thống kê/tiền xử lý.

In [ ]:
!unzip -q -n /content/drive/MyDrive/CAMUS_public.zip -d /content/data/camus && ls /content/data/camus

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python data_camus.py --src /content/data/camus

## Cell 5 (Bước 1C). CholecSeg8k (Kaggle)
1. Tự đăng nhập Kaggle, mở trang dataset *CholecSeg8k*, đồng ý điều khoản nếu có.
2. Tự tạo API token (Kaggle → Settings → Create New Token). Chạy cell, khi hiện ô nhập thì dán token vào (ô ẩn ký tự, token không lưu trong notebook; đừng dán vào chat).
3. Script in dung lượng (~2,9 GB) và chờ bạn gõ `yes`. Tên dataset đúng là `newslab/cholecseg8k`.

In [ ]:
import os, getpass
if not os.environ.get('KAGGLE_API_TOKEN'): os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Dán Kaggle API token rồi nhấn Enter: ')
%cd /content/drive/MyDrive/anatomy_project/src
!python data_cholec.py --download --slug newslab/cholecseg8k

## Cell 6 (Bước 2). TotalSegmentator trên 5 ca test CT
(Tải weights của TotalSegmentator lần đầu, vài trăm MB đến ~1-2 GB từ nguồn chính thức. Nếu hết VRAM: thêm `--fast`.)

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python totalseg_baseline.py --n 5

## Cell 7 (Bước 3). Huấn luyện 3 mô hình (30 phút mỗi mô hình, đổi bằng `--budget-min`)
Mỗi cell chạy độc lập, kết quả lưu ngay vào Drive.

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python train_ct.py --budget-min 30

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python train_2d.py --name camus --budget-min 30

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python train_2d.py --name cholec --budget-min 30

## Cell 8 (Bước 4). Đo tốc độ

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python speed.py --name camus
!python speed.py --name cholec
!python speed.py --name ct

## Cell 9 (Bước 5). Gom bảng số liệu
Tạo `results/REPORT_DATA.md`. Sau đó tải thư mục `results/` về Mac (hoặc báo Claude) để viết nhận xét, lỗi, hạn chế, đề xuất.

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python report_tables.py

## Tuần 3-4. Mô hình chính xác + cải tiến + ablation
Chạy sau Cell 7 (đã có baseline và checkpoint tuần 2). Chi tiết và cách đọc kết quả: `TUAN3-4.md`. Chạy lại Cell 1 nếu phiên bị ngắt.
### Cell 10 (Tuần 3). Phân tích lỗi mô hình 2D gốc

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python analyze_2d.py errors --name camus
!python analyze_2d.py errors --name cholec


### Cell 11 (Tuần 3). nnU-Net 3D trên CT (đúng chia tập đã có). `prepare` và `train` lâu; `train` chạy được tiếp bằng `--resume`

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python nnunet_ct.py prepare
!python nnunet_ct.py train --trainer nnUNetTrainer_50epochs
!python nnunet_ct.py predict
!python nnunet_ct.py evaluate


### Cell 12 (Tuần 4). Ablation 2D: mỗi cấu hình đổi MỘT thứ so với baseline (cùng 30 phút như baseline để so công bằng)

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
for ds in ['camus', 'cholec']:
    !python train_2d.py --name {ds} --budget-min 30 --tag loss_focal --loss dice_focal
    !python train_2d.py --name {ds} --budget-min 30 --tag aug_strong --aug strong
    !python train_2d.py --name {ds} --budget-min 30 --tag enc_r34 --encoder resnet34
    !python train_2d.py --name {ds} --budget-min 30 --tag no_pretrain --no-pretrain


### Cell 13 (Tuần 4). Ensemble + bảng ablation (đổi `--tags` theo cấu hình tốt nhất ở bảng)

In [ ]:
%cd /content/drive/MyDrive/anatomy_project/src
!python analyze_2d.py ensemble --name camus --tags loss_focal aug_strong enc_r34
!python analyze_2d.py ensemble --name cholec --tags loss_focal aug_strong enc_r34
!python analyze_2d.py ablation --name camus
!python analyze_2d.py ablation --name cholec
!python report_tables.py
